# Chapter 4 — The Network (without nn.Module)

**Book alignment:** PyTorch From First Principles, Chapter 4

**Question this notebook isolates:** Does stacking two affine layers without a nonlinearity collapse exactly to one affine map that cannot solve sign-XOR, while inserting ReLU makes it solvable (>90% held-out accuracy)?


In [ ]:
import math
import numpy as np
import torch

torch.manual_seed(0)
np.random.seed(0)

n = 800
X = torch.randn(n, 2)
y = ((X[:, 0] > 0) ^ (X[:, 1] > 0)).long()
perm = torch.randperm(n)
X_train, y_train = X[perm[:640]], y[perm[:640]]
X_val, y_val = X[perm[640:]], y[perm[640:]]
D_in, H, C = 2, 16, 2


## 1. Two affine layers collapse to one — exactly

`(X @ W1 + b1) @ W2 + b2 = X @ (W1 @ W2) + (b1 @ W2 + b2)`. The max difference must sit at the float32 noise floor.


In [ ]:
torch.manual_seed(1)
W1 = (torch.randn(D_in, H) / math.sqrt(D_in)).requires_grad_()
b1 = torch.zeros(H, requires_grad=True)
W2 = (torch.randn(H, C) / math.sqrt(H)).requires_grad_()
b2 = torch.zeros(C, requires_grad=True)

with torch.no_grad():
    W_eq = W1 @ W2
    b_eq = b1 @ W2 + b2
    two_layer = (X_train @ W1 + b1) @ W2 + b2
    one_layer = X_train @ W_eq + b_eq
    gap = (two_layer - one_layer).abs().max().item()
print(f"W_eq={tuple(W_eq.shape)} b_eq={tuple(b_eq.shape)} max|diff|={gap:.3e}")


In [ ]:
assert tuple(W_eq.shape) == (2, 2) and tuple(b_eq.shape) == (2,)
assert gap < 1e-4, gap
print("82 parameters expressing what 6 could: depth without nonlinearity adds nothing")


## 2. Black-box test: the affine additivity identity

Affine `f` satisfies `f(x1+x2) - f(0) = (f(x1)-f(0)) + (f(x2)-f(0))`. The no-activation net must pass at float precision; the ReLU net must violate it by orders of magnitude.


In [ ]:
def affine_residual(f, x1, x2):
    with torch.no_grad():
        zero = torch.zeros_like(x1)
        lhs = f(x1 + x2) - f(zero)
        rhs = (f(x1) - f(zero)) + (f(x2) - f(zero))
    return (lhs - rhs).abs().max().item()

a = torch.randn(64, 2)
b = torch.randn(64, 2)
r_lin = affine_residual(lambda x: (x @ W1 + b1) @ W2 + b2, a, b)
r_relu = affine_residual(lambda x: torch.relu(x @ W1 + b1) @ W2 + b2, a, b)
print(f"no activation residual={r_lin:.3e}")
print(f"with ReLU residual={r_relu:.4f}")


In [ ]:
assert r_lin < 1e-4, r_lin
assert r_relu > 0.1, r_relu
print("identity holds for the affine stack, fails once ReLU breaks affinity")


## 3. Representation decides: linear stalls near chance, ReLU solves XOR

Same data, seeds, lr, and steps. The linear stack must stall well below 75% train accuracy while the ReLU net exceeds 90% on held-out data.


In [ ]:
def make_params(seed=1):
    g = torch.Generator().manual_seed(seed)
    W1 = (torch.randn(D_in, H, generator=g) / math.sqrt(D_in)).requires_grad_()
    b1 = torch.zeros(H, requires_grad=True)
    W2 = (torch.randn(H, C, generator=g) / math.sqrt(H)).requires_grad_()
    b2 = torch.zeros(C, requires_grad=True)
    return [W1, b1, W2, b2]

def run(use_relu, steps=600, lr=0.5):
    ps = make_params()
    W1, b1, W2, b2 = ps
    for _ in range(steps):
        logits = (torch.relu(X_train @ W1 + b1) if use_relu else (X_train @ W1 + b1)) @ W2 + b2
        logp = torch.log_softmax(logits, dim=1)
        loss = -logp[torch.arange(y_train.shape[0]), y_train].mean()
        for p in ps:
            if p.grad is not None:
                p.grad.zero_()
        loss.backward()
        with torch.no_grad():
            for p in ps:
                p -= lr * p.grad
    with torch.no_grad():
        tr = (((torch.relu(X_train @ W1 + b1) if use_relu else (X_train @ W1 + b1)) @ W2 + b2).argmax(1) == y_train).float().mean().item()
        va = (((torch.relu(X_val @ W1 + b1) if use_relu else (X_val @ W1 + b1)) @ W2 + b2).argmax(1) == y_val).float().mean().item()
    return tr, va

tr_lin, va_lin = run(False)
tr_relu, va_relu = run(True)
print(f"linear: train={tr_lin:.3f} val={va_lin:.3f}")
print(f"relu:   train={tr_relu:.3f} val={va_relu:.3f}")


In [ ]:
assert tr_lin < 0.75, tr_lin
assert va_relu > 0.90, va_relu
assert va_relu > va_lin + 0.2
print("nonlinearity enlarges the function class; optimization alone cannot")


## What we earned

A network of raw tensors is parameters plus a function combining them — and depth only adds representational power once a nonlinearity prevents algebraic collapse. Both runs lowered their loss; only held-out accuracy plus the collapse evidence told them apart.

Chapter 5 hands parameter bookkeeping to `nn.Module` and asks which tensors PyTorch actually considers part of the model.
